In [8]:
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
import h5py
import os,json
from pathlib import Path
from pprint import pprint
import dask.dataframe as dd

### 打印看看里面包括哪些指标


In [2]:
fname = r"/Users/a1-6/Downloads/GEDI02_A_002-20251009_201109/GEDI02_A_2022338025411_O22522_02_T03572_02_003_02_V002.h5"

with h5py.File(fname, 'r') as f:
    def print_tree(name, obj):
        print(name)
    f.visititems(print_tree)

BEAM0000
BEAM0000/ancillary
BEAM0000/ancillary/l2a_alg_count
BEAM0000/beam
BEAM0000/channel
BEAM0000/degrade_flag
BEAM0000/delta_time
BEAM0000/digital_elevation_model
BEAM0000/digital_elevation_model_srtm
BEAM0000/elev_highestreturn
BEAM0000/elev_lowestmode
BEAM0000/elevation_bias_flag
BEAM0000/elevation_bin0_error
BEAM0000/energy_total
BEAM0000/geolocation
BEAM0000/geolocation/elev_highestreturn_a1
BEAM0000/geolocation/elev_highestreturn_a2
BEAM0000/geolocation/elev_highestreturn_a3
BEAM0000/geolocation/elev_highestreturn_a4
BEAM0000/geolocation/elev_highestreturn_a5
BEAM0000/geolocation/elev_highestreturn_a6
BEAM0000/geolocation/elev_lowestmode_a1
BEAM0000/geolocation/elev_lowestmode_a2
BEAM0000/geolocation/elev_lowestmode_a3
BEAM0000/geolocation/elev_lowestmode_a4
BEAM0000/geolocation/elev_lowestmode_a5
BEAM0000/geolocation/elev_lowestmode_a6
BEAM0000/geolocation/elev_lowestreturn_a1
BEAM0000/geolocation/elev_lowestreturn_a2
BEAM0000/geolocation/elev_lowestreturn_a3
BEAM0000/geoloca

In [15]:
def read_GEDI(fname, poly=None, urban_thresh=0.5):

    all_beams = []

    with h5py.File(fname, 'r') as fi:
        beam_groups = [k for k in fi.keys() if k.startswith("BEAM")]

        for g in beam_groups:
            grp = fi[g]

            def _opt(*candidates):
                for p in candidates:
                    if p in grp:
                        return p
                return None

            lat_path = _opt("lat_lowestmode", "lat_highestreturn")
            lon_path = _opt("lon_lowestmode", "lon_highestreturn")
            if lat_path is None or lon_path is None:
                continue

            lat = grp[lat_path][:]
            lon = grp[lon_path][:]

            def take(relpath):
                if relpath is None:
                    return np.full(lat.shape, np.nan)
                if relpath in grp:
                    return grp[relpath][:]
                # 处理子目录路径（如 "land_cover_data/urban_proportion"）
                try:
                    ds = grp
                    for part in relpath.split('/'):
                        ds = ds[part]
                    return ds[:]
                except Exception:
                    return np.full(lat.shape, np.nan)

            # 字段映射
            elev_lowestmode_best = _opt("elev_lowestmode_a2", "elev_lowestmode_a1", "elev_lowestmode")
            h_te_best_fit = take(elev_lowestmode_best)         # “最佳拟合”地表高程
            h_te_interp   = take("elev_lowestmode")            # 近似当作 “h_te_interp”
            h_te_mean     = take(_opt("rh100", "elev_highestreturn"))

            terrain_flg   = take("l2_quality_flag")            # 1=好；0=差
            degrade_flag  = take("degrade_flag")               # 0 正常；1 退化
            urban_prop = take("land_cover_data/urban_proportion")
            with np.errstate(invalid='ignore'):
                urban_flag = (urban_prop > float(urban_thresh)).astype(np.uint8)

            n_te_photons  = take("sensitivity")                # 探测能力近似

            # 这些 GEDI 未直接提供，留 NaN
            dem_h         = np.full_like(h_te_interp, np.nan, dtype=float)
            terrain_slope = np.full_like(h_te_interp, np.nan, dtype=float)

            df = pd.DataFrame({
                "beam": g,
                "lat": lat,
                "lon": lon,
                "dem_h": dem_h,
                "h_te_interp": h_te_interp,
                "h_te_mean": h_te_mean,
                "h_te_best_fit": h_te_best_fit,
                "terrain_flg": terrain_flg,
                "urban_flag": urban_flag,
                "cloud_flag_atm": degrade_flag,   # 近似用 degrade_flag 代替
                "terrain_slope": terrain_slope,
                "n_te_photons": n_te_photons,  # 原生字段保留
                "degrade_flag": degrade_flag,
                "urban_proportion": urban_prop
            })

            all_beams.append(df)

    if not all_beams:
        return pd.DataFrame()

    df_all = pd.concat(all_beams, ignore_index=True)

    if poly is not None and len(df_all) > 0:
        from shapely.geometry import Point
        from shapely.geometry import GeometryCollection
        gdf = gpd.GeoDataFrame(
            df_all,
            geometry=gpd.points_from_xy(df_all["lon"].values, df_all["lat"].values),
            crs="EPSG:4326"
        )

        if hasattr(poly, "geometry"):  # GeoDataFrame
            poly_gs = poly.geometry
            poly_crs = getattr(poly, "crs", None)
        elif hasattr(poly, "crs"):     # GeoSeries
            poly_gs = poly
            poly_crs = poly.crs
        else:
            if isinstance(poly, (list, tuple)):
                from shapely.ops import union_all
                poly_union = union_all(poly) if len(poly) > 0 else GeometryCollection()
            else:
                poly_union = poly
            poly_gs = gpd.GeoSeries([poly_union], crs="EPSG:4326")
            poly_crs = poly_gs.crs

        # 投影到与点一致的 CRS
        if poly_crs is not None and poly_crs != gdf.crs:
            poly_gs = poly_gs.to_crs(gdf.crs)

        # 使用 clip（内部做相交）
        gdf_clip = gpd.clip(gdf, poly_gs.union_all())
        # 丢掉 geometry 返回 DataFrame
        df_all = pd.DataFrame(gdf_clip.drop(columns="geometry"))

    return df_all

In [16]:
poly = gpd.read_file("/Users/a1-6/Self_program/underground_water/boundary/Poyang_Lakes1.shp")
file = "/Users/a1-6/Self_program/underground_water/GEDI/GEDI02_A_002-20251010_082407/GEDI02_A_2022338025411_O22522_02_T03572_02_003_02_V002.h5"
df = read_GEDI(file, poly=poly)
print(df.head())
print(df.columns.tolist())
print(len(df), "shots")

             beam        lat         lon  dem_h  h_te_interp    h_te_mean  \
1239126  BEAM1011  29.079340  115.934495    NaN  2071.914551  2071.914551   
1239127  BEAM1011  29.079718  115.934894    NaN  2072.227051  2072.227051   
1239128  BEAM1011  29.080096  115.935294    NaN  2072.015137  2072.015137   
1239129  BEAM1011  29.080482  115.935700    NaN  2100.201416  2100.201416   
1239130  BEAM1011  29.080861  115.936100    NaN  2101.863281  2101.863281   

         h_te_best_fit  terrain_flg  urban_flag  cloud_flag_atm  \
1239126    2071.914551          NaN           0               0   
1239127    2072.227051          NaN           0               0   
1239128    2072.015137          NaN           0               0   
1239129    2100.201416          NaN           0               0   
1239130    2101.863281          NaN           0               0   

         terrain_slope  n_te_photons  degrade_flag  urban_proportion  
1239126            NaN      0.571292             0             